# **A/B Testing Analysis of a Marketing Campaing** 

## **Conduct a two-proportion Z-test in order to check wether exposure of users to adds leads to increase in conversions.**

If we define conversion rate for both treatment and control group as:<br/>
**Treatment group:** $p_1 = x_1 / n_1$, where $x_1$ is the number of conversions and $n_1$ - the number of participants in the treatment group.<br/> 
**Control group**: $p_2 = x_2 / n_2$, where $x_2$ is the number of conversions and $n_2$ -the number of participants in the control group.

**Null Hypothesis ($H_0$):** $p_1 = p_2$ and any increase in conversions due to treament is random. <br/>
**Alternative Hypothesis ($H_1$):** $p_1 > p_2$, so the treatment do increase conversion rate. 

In [1]:
import os
import pandas as pd

DATASETS_PATH = os.getenv("DATASETS_PATH")
DATASETS_PATH = DATASETS_PATH + "/MarketingA_B_Testing/"

data_df = pd.read_csv(DATASETS_PATH + "marketing_AB.csv")

# Calculate sample conversion rates:
conversions = data_df.groupby("test group")["converted"].value_counts(normalize=True)

p1 = conversions["ad"][True]
p2 = conversions["psa"][True]

print("p1 = "+ str(p1))
print()
print("p2 = "+ str(p2))


p1 = 0.025546559636683747

p2 = 0.01785410644448223


In [2]:
#Calculate pooled conversion rate:
conversion = data_df["converted"].value_counts(normalize=True)

p = conversion[True]
print("p = " + str(p))

p = 0.02523886203220195


In [3]:
# Calculate number of users in each group
n_users = data_df["test group"].value_counts()

n_users_1 = n_users["ad"]
n_users_2 = n_users["psa"]

print("Number of users in treatment group: " + str(n_users_1))
print("Number of users in control group: " + str(n_users_2))

Number of users in treatment group: 564577
Number of users in control group: 23524


In [4]:
# Calculate standard error:
import numpy as np
se = np.sqrt((p*(1-p)*((1.0/n_users_1) + (1.0/n_users_2))))
print("Standard Error: " + str(se))

Standard Error: 0.0010437410649006525


In [5]:
# Calculate Z-score:
z_score = (p1 - p2)/se
print("Z-score: " + str(z_score))

Z-score: 7.3700781265454145


In [6]:
from scipy.stats import norm

# one-sided
p_value = 1 - norm.cdf(z_score)

print("P-value: " + str(p_value))

P-value: 8.526512829121202e-14


Due to very small p-value (< 0.05), there is a very strong statistical evidence for alternative hypothesis and thus the treatment does improve conversions.

## **Calculate the treatment lift and confidence interval for it.**

In [7]:
delta = p1 - p2

se = np.sqrt(
    (p1 * (1 - p1)) / n_users_1 +
    (p2 * (1 - p2)) / n_users_2
)

z = norm.ppf(0.975)  # 95% CI -> 2.5% in each tail (total 5%), using percent point function (the inverse of CDF)

lower = delta - z * se # comes from the z-score
upper = delta + z * se

print(f"Lift: {delta:.6f}")
print(f"95% CI: [{lower:.6f}, {upper:.6f}]")

Lift: 0.007692
95% CI: [0.005951, 0.009434]


The baseline conversion rate is $\approx 1.79$% (in the control group), so the relative lift of the treatment is around $43$%.

## **Check the Null Hypothesis by Bootstrapping.**

In [8]:
ad = data_df[data_df["test group"] == "ad"]["converted"].values
psa = data_df[data_df["test group"] == "psa"]["converted"].values

combined = np.concatenate([ad, psa])
n1, n2 = len(ad), len(psa)

obs_diff = ad.mean() - psa.mean()

print("Observed difference in conversion rates: " + str(obs_diff))

number_of_experiments = 10000
diffs = []

B = number_of_experiments
combined = np.array(combined)

n = len(combined)
p = n1 / n

# random assignment matrix (B x n)
mask = np.random.rand(B, n) < p

# broadcast data without tiling
group1_means = (mask * combined).sum(axis=1) / mask.sum(axis=1)
group2_means = ((~mask) * combined).sum(axis=1) / (~mask).sum(axis=1)

diffs = group1_means - group2_means

p_value = np.mean(diffs >= obs_diff)

print("P-Value: " + str(p_value))

Observed difference in conversion rates: 0.007692453192201517
P-Value: 0.0


The bootstrapping method delivers the strong evidence too, that the null hypothesis is wrong and the treatment does have effect on conversion rates. 

## **Calculate Confidence Interval for the Difference of Conversion Rates in Groups.**

In [9]:
# vectorized resampling (B x n1, B x n2)
boot_ad = np.random.choice(ad, size=(B, n1), replace=True)
boot_psa = np.random.choice(psa, size=(B, n2), replace=True)

# compute means along rows
boot_ad_means = boot_ad.mean(axis=1)
boot_psa_means = boot_psa.mean(axis=1)

# bootstrap distribution of differences
boot_diffs = boot_ad_means - boot_psa_means

# confidence interval
lower = np.percentile(boot_diffs, 2.5)
upper = np.percentile(boot_diffs, 97.5)

print("95% CI:", (lower.item(), upper.item()))

95% CI: (0.005993700952547869, 0.009410600920082116)


The ruslt of the bootstrap calculation is very close to the confidents interval from Z-statistics above.

## **Indermidiate conclusions:**
There is a very strong statistical evidence delivered by analytical and numerical methods, that there is an effect of user treatment by adds leading to increase in conversion rate. The realtive conversion rate lift is about $43$ % laying in a $95$ % confidence interval between $33$ % and $53$ %.

## **Analysis of the total adds value effect on conversion rate**

$$
logit(p) = \beta_0 + \beta_1*\text{ads} + \beta_2*\text{group} + \beta_3*(\text{ads}*\text{group})
$$

In [10]:
import statsmodels.api as sm

data_df["group"] = (data_df["test group"] == "ad").astype(int)
data_df["interaction"] = data_df["total ads"] * data_df["group"]

X = data_df[["total ads", "group", "interaction"]]
y = data_df["converted"].astype(int)

X = sm.add_constant(X)

model = sm.Logit(y, X)
result = model.fit()

print(result.summary())

Optimization terminated successfully.
         Current function value: 0.108986
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:              converted   No. Observations:               588101
Model:                          Logit   Df Residuals:                   588097
Method:                           MLE   Df Model:                            3
Date:                Thu, 23 Apr 2026   Pseudo R-squ.:                 0.07467
Time:                        18:11:39   Log-Likelihood:                -64095.
converged:                       True   LL-Null:                       -69267.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
const          -4.4035      0.059    -74.624      0.000      -4.519      -4.288
total ads       0.0097    

In [11]:
psa_model = sm.Logit(
    data_df[data_df["group"] == 0]["converted"],
    sm.add_constant(data_df[data_df["group"] == 0]["total ads"])
).fit()
print(psa_model.summary())

Optimization terminated successfully.
         Current function value: 0.083277
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:              converted   No. Observations:                23524
Model:                          Logit   Df Residuals:                    23522
Method:                           MLE   Df Model:                            1
Date:                Thu, 23 Apr 2026   Pseudo R-squ.:                 0.07022
Time:                        18:23:28   Log-Likelihood:                -1959.0
converged:                       True   LL-Null:                       -2106.9
Covariance Type:            nonrobust   LLR p-value:                 2.581e-66
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -4.4035      0.059    -74.624      0.000      -4.519      -4.288
total ads      0.0097      0.

While the A/B test provides a valid estimate of the causal effect of the advertising treatment (AD vs PSA), the observed relationship between total ads and conversion cannot be interpreted causally in either group, as ad exposure is not guaranteed to be randomly assigned and is likely influenced by underlying user behavior and platform targeting mechanisms.